# Clean data

Run this notebook before **EDA**. It reads the eight February and December TLC files one at a time, cleans card-paid trips, and writes a reproducible random sample of 250,000 trips per month. February and December 2022–2024 are for development and 2025 is reserved for later-year model evaluation.

Run from the repository root or the notebooks folder with pandas, PyArrow, and Matplotlib installed.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data"
PROCESSED = RAW / "processed"
SAMPLE = RAW / "sample"
for folder in (PROCESSED, SAMPLE):
    folder.mkdir(parents=True, exist_ok=True)

MONTHS = ["2022-02", "2022-12", "2023-02", "2023-12",
          "2024-02", "2024-12", "2025-02", "2025-12"]
N_PER_MONTH = 250_000
SEED = 42

AIRPORT_ZONES = {1, 132, 138}       # Newark, JFK, LaGuardia
UNKNOWN_ZONES = {264, 265}          # Unknown / Outside of NYC

# Holiday periods used for the holiday flag
DEC_HOLIDAY_START_DAY = 24          # Dec 24-31
PRESIDENTS_DAY = {2022: "2022-02-21", 2023: "2023-02-20",
                  2024: "2024-02-19", 2025: "2025-02-17"}

DIST_BINS = [0, 1, 3, 7, 15, np.inf]
DIST_LABELS = ["0-1", "1-3", "3-7", "7-15", "15+"]

# cleaning rules
RULES = {
    "min_duration": 1,      # minutes
    "max_duration": 24 * 60,  # minutes
    "max_speed": 80,        # mph
}

KEEP_COLS = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
             "passenger_count", "trip_distance", "RatecodeID",
             "PULocationID", "DOLocationID", "payment_type", "fare_amount",
             "extra", "mta_tax", "tip_amount", "tolls_amount",
             "improvement_surcharge", "total_amount", "congestion_surcharge",
             "airport_fee", "cbd_congestion_fee"]

pd.Series(RULES, name="Cleaning threshold")

min_duration       1
max_duration    1440
max_speed         80
Name: Cleaning threshold, dtype: int64

## Cleaning and feature functions

In [2]:
def load_month(month):
    """Read one raw month and give every year the same column names."""
    df = pd.read_parquet(RAW / f"yellow_tripdata_{month}.parquet")
    df = df.rename(columns={"Airport_fee": "airport_fee"})
    if "cbd_congestion_fee" not in df.columns:      # only exists from 2025
        df["cbd_congestion_fee"] = np.nan
    return df[KEEP_COLS]


def clean_month(df, month):
    """Apply the cleaning rules in order.

    Returns the cleaned card-paid trips and a list of (rule, rows_left)
    so we can build the cleaning funnel.
    """
    funnel = [("raw rows", len(df))]

    start = pd.Timestamp(month + "-01")
    end = start + pd.offsets.MonthBegin(1)
    df = df[(df.tpep_pickup_datetime >= start) & (df.tpep_pickup_datetime < end)]
    funnel.append(("pickup inside the month", len(df)))

    df = df[df.payment_type == 1]
    funnel.append(("card-paid (payment_type = 1)", len(df)))

    df = df[np.isfinite(df.fare_amount) & (df.fare_amount > 0)]
    funnel.append(("positive finite fare", len(df)))

    df = df[np.isfinite(df.tip_amount) & (df.tip_amount >= 0)]
    funnel.append(("nonnegative finite tip", len(df)))

    df = df[np.isfinite(df.trip_distance) & (df.trip_distance > 0)]
    funnel.append(("positive finite distance", len(df)))

    dur = (df.tpep_dropoff_datetime - df.tpep_pickup_datetime).dt.total_seconds() / 60
    df = df.assign(duration_min=dur)
    df = df[(df.duration_min >= RULES["min_duration"]) &
            (df.duration_min <= RULES["max_duration"])]
    funnel.append((f"{RULES['min_duration']}-{RULES['max_duration']} min duration", len(df)))

    df = df.assign(speed_mph=df.trip_distance / (df.duration_min / 60))
    df = df[df.speed_mph <= RULES["max_speed"]]
    funnel.append((f"speed <= {RULES['max_speed']} mph", len(df)))

    df = df[~df.PULocationID.isin(UNKNOWN_ZONES) & ~df.DOLocationID.isin(UNKNOWN_ZONES)]
    funnel.append(("known pickup & drop-off zone", len(df)))

    pre_tip_charge = df["total_amount"] - df["tip_amount"]
    df = df.loc[np.isfinite(pre_tip_charge) & (pre_tip_charge > 0)]
    funnel.append(("valid full pre-tip charge", len(df)))

    return df, funnel


def load_zones():
    return pd.read_csv(RAW / "taxi_zone_lookup.csv")

In [3]:
def add_features(df, zones):
    """Time, trip and location features (no weather for now)."""
    df = df.copy()
    t = df.tpep_pickup_datetime
    df["year"] = t.dt.year
    df["month_num"] = t.dt.month
    df["month_name"] = np.where(df.month_num == 2, "Feb", "Dec")
    df["day"] = t.dt.day
    df["date"] = t.dt.normalize()
    df["hour"] = t.dt.hour
    df["dow"] = t.dt.dayofweek                       # 0 = Monday
    df["is_weekend"] = df.dow >= 5
    df["is_night"] = (df.hour >= 22) | (df.hour < 6)

    # holiday flag: Dec 24-31, or Presidents' Day long weekend (Sat-Mon)
    pres = df.year.map(PRESIDENTS_DAY).pipe(pd.to_datetime)
    in_pres = (df.month_num == 2) & (df.date >= pres - pd.Timedelta(days=2)) & (df.date <= pres)
    in_xmas = (df.month_num == 12) & (df.day >= DEC_HOLIDAY_START_DAY)
    df["is_holiday"] = in_pres | in_xmas

    df["is_airport"] = df.PULocationID.isin(AIRPORT_ZONES) | df.DOLocationID.isin(AIRPORT_ZONES)
    df["same_zone"] = df.PULocationID == df.DOLocationID
    df["dist_bucket"] = pd.cut(df.trip_distance, DIST_BINS, labels=DIST_LABELS, right=False)

    df["tip_pct"] = 100 * df.tip_amount / df.fare_amount
    # Tip as a percentage of everything paid except the tip.
    pre_tip = df.total_amount - df.tip_amount
    df["tip_pct_total"] = np.where(pre_tip > 0, df.tip_amount / pre_tip * 100, np.nan)
    df["tip_over_100_pct"] = df.tip_amount > pre_tip
    df["paid_cbd_fee"] = df.cbd_congestion_fee.fillna(0) > 0   # congestion pricing (2025 only)
    df["zero_tip"] = df.tip_amount == 0
    df["tip_20plus"] = df.tip_pct >= 20

    z = zones.set_index("LocationID")
    df["pu_borough"] = df.PULocationID.map(z.Borough)
    df["pu_zone"] = df.PULocationID.map(z.Zone)
    df["pu_service_zone"] = df.PULocationID.map(z.service_zone)
    df["do_borough"] = df.DOLocationID.map(z.Borough)

    # smaller dtypes to save memory
    for c in ["trip_distance", "fare_amount", "tip_amount", "tip_pct", "tip_pct_total",
              "duration_min", "speed_mph", "total_amount", "tolls_amount"]:
        df[c] = df[c].astype("float32")
    return df

## Clean and sample each month

Keep card-paid trips with a pickup in the source month, a positive fare and distance, a nonnegative tip, and known pickup and drop-off zones. The one-minute minimum, 24-hour maximum, and 80 mph average-speed ceiling screen for implausible records. Trips also need a positive, finite pre-tip charge. The cleaning log reports how many rows each filter removes.

The primary outcome is the recorded tip as a percentage of the full pre-tip charge. Trips above 100% of that charge stay in the sample with a flag for the EDA sensitivity check.

The two EDA CSVs contain counts that sampling would obscure. Payment counts describe raw records by file. Daily volume counts describe all eligible cleaned card-paid trips. The trip-level Parquet files supply the tipping charts and future models.

In [4]:
from IPython.display import display

zones = load_zones()
payment_labels = {1: "Card", 2: "Cash", 0: "Code 0"}
payment_counts = []
daily_counts = []
cleaning_log = {}

for month in MONTHS:
    raw = load_month(month)
    payment = raw["payment_type"].map(payment_labels).fillna("Other or missing")
    counts = payment.value_counts().rename_axis("payment").reset_index(name="trips")
    payment_counts.append(counts.assign(month=month))

    clean, funnel = clean_month(raw, month)
    del raw
    dates = clean["tpep_pickup_datetime"].dt.normalize()
    calendar = pd.date_range(month, periods=pd.Period(month).days_in_month)
    daily = dates.value_counts().reindex(calendar, fill_value=0)
    daily = daily.rename_axis("date").reset_index(name="trips")
    daily_counts.append(daily.assign(month=month))

    sample = clean.sample(n=N_PER_MONTH, random_state=SEED)
    sample = add_features(sample, zones)
    sample.to_parquet(SAMPLE / f"sample_{month}.parquet", index=False)
    funnel.append(("sampled trips", len(sample)))
    cleaning_log[month] = dict(funnel)
    del clean, sample

pd.concat(payment_counts, ignore_index=True).to_csv(
    PROCESSED / "eda_payment_counts.csv", index=False
)
pd.concat(daily_counts, ignore_index=True).to_csv(
    PROCESSED / "eda_daily_volume.csv", index=False
)

funnel_table = pd.DataFrame(cleaning_log)
distance_exclusions = (
    funnel_table.loc["nonnegative finite tip"] - funnel_table.loc["positive finite distance"]
).rename("Excluded at the distance step")
display(distance_exclusions.to_frame())
funnel_table

,Excluded at the distance step
2022-02,17457
2022-12,26868
2023-02,19430
2023-12,22476
2024-02,19619
2024-12,24335
2025-02,19029
2025-12,24349


,2022-02,2022-12,2023-02,2023-12,2024-02,2024-12,2025-02,2025-12
raw rows,2979431,3399549,2913955,3376567,3007526,3668371,3577543,4305006
pickup inside the month,2979367,3399503,2913900,3376524,3007511,3668337,3577512,4304997
card-paid (payment_type = 1),2296167,2553771,2309545,2574541,2341998,2733337,2336147,2618764
positive finite fare,2295734,2553401,2309283,2574322,2341850,2733245,2336079,2618451
nonnegative finite tip,2295734,2553401,2309283,2574322,2341850,2733245,2336079,2618451
positive finite distance,2278277,2526533,2289853,2551846,2322231,2708910,2317050,2594102
1-1440 min duration,2274984,2520951,2286882,2547577,2318799,2703593,2309701,2539453
speed <= 80 mph,2274910,2520848,2286809,2547517,2318754,2703471,2309648,2539363
known pickup & drop-off zone,2246508,2471613,2249784,2521963,2299060,2679544,2292145,2519430
valid full pre-tip charge,2246508,2471613,2249784,2521963,2299060,2679544,2292145,2519430
